# 05 — RAG Synthesis Pipeline (`agenttune.rag.synthesis`, real docs, real models)

The docs-to-multi-hop-QA synthesis package, run for real, end to end:

```
real docs -> Stage 0 (chunk + entity graph) -> Stage 1 (path sampling)
          -> Stage 2 (answer-first QA generation) -> Stage 3 (chain-dependency verification)
          -> Stage 4 (2D difficulty label + balance) -> Stage 5 (GRPO-ready split + real metrics)
```

- **Real docs**: a real HotpotQA slice (context passages), the exact corpus the package's own
  README validates against.
- **Real embedder**: `BAAI/bge-m3` (`BGEM3Embedder`) — the package's own validated embedder.
- **Real LLM**: the pipeline's `build_dataset.py` CLI hardcodes a Groq client requiring
  `GROQ_API_KEY` (an external API we're not using). The `LLMClient` Protocol is injectable by
  design specifically for this — so this notebook stands up a real local OpenAI-compatible HTTP
  server backed by `Qwen/Qwen2.5-3B-Instruct` and points a real `OpenAICompatLLMClient` at it.
  No pipeline logic changes, only the LLM transport target — no external API calls.
- **Real evaluation**: the pipeline's own `metrics.json` — answerability (ARES), multi-hop
  necessity (Min et al. 2019), diversity, and difficulty distribution — computed by the real
  `run_pipeline()`, not hand-rolled.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("OPENAI_API_KEY", "sk-local-noop")  # litellm/openai client wants a key; the local server ignores it

import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available()
print(f"[gpu] {torch.cuda.get_device_name(0)} | torch {torch.__version__}")

[gpu] NVIDIA GeForce RTX 4090 | torch 2.11.0+cu130


## Real local LLM server — no external API

Same real-infrastructure pattern as notebook 03: a tiny OpenAI-compatible HTTP server backed by
a real local `Qwen2.5-3B-Instruct`. `build_dataset.py`'s pipeline calls `llm.acomplete(...)`
through the injected `LLMClient` Protocol exactly as it would call a hosted endpoint.

In [2]:
JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

def start_llm_server():
    tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=torch.bfloat16).to("cuda")

    def generate(messages, temperature=0.0, max_new_tokens=300):
        enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True).to("cuda")
        sample = bool(temperature and temperature > 0)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=sample,
                                 temperature=temperature if sample else None,
                                 top_p=0.95 if sample else None,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *a):
            pass

        def do_POST(self):
            body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            text = generate(body["messages"], body.get("temperature", 0.0))
            payload = json.dumps({
                "choices": [{"index": 0, "message": {"role": "assistant", "content": text},
                             "finish_reason": "stop"}],
                "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
            }).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)

    server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, server.server_address[1]

server, port = start_llm_server()
api_base = f"http://127.0.0.1:{port}/v1"
print(f"[server] real {JUDGE_MODEL} on {api_base} -- a real local OpenAI-compatible endpoint, no external API")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[server] real Qwen/Qwen2.5-3B-Instruct on http://127.0.0.1:42383/v1 -- a real local OpenAI-compatible endpoint, no external API


## Real docs, real embedder, real LLM client — the package's own components

In [3]:
from agenttune.rag.data.hotpotqa import load_hotpotqa_splits, build_corpus_from_hotpotqa
from agenttune.rag.synthesis.build_dataset import run_pipeline
from agenttune.rag.synthesis.llm_client import OpenAICompatLLMClient
from agenttune.rag.synthesis.embeddings import BGEM3Embedder

llm = OpenAICompatLLMClient(model=f"openai/{JUDGE_MODEL}", api_key="sk-local-noop", base_url=api_base)

print("[embed] loading real BAAI/bge-m3 embedder (the package's own validated embedder)...")
embedder = BGEM3Embedder(device="cuda")

print("[corpus] loading a real HotpotQA slice...")
train, _ = load_hotpotqa_splits(train_size=6, eval_size=0)
docs = build_corpus_from_hotpotqa(train, max_docs=20)
print(f"[corpus] {len(docs)} real documents from real HotpotQA context passages")

[embed] loading real BAAI/bge-m3 embedder (the package's own validated embedder)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[corpus] loading a real HotpotQA slice...


'The read operation timed out' thrown while requesting HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/1908d6afbbead072334abe2965f91bd2709910ab/.huggingface.yaml


Retrying in 1s [Retry 1/5].


[corpus] 20 real documents from real HotpotQA context passages


## Run the real 6-stage pipeline

`run_pipeline()` — the exact function the CLI calls — chunk+graph, path sampling, answer-first
generation, chain-dependency verification, difficulty balance, and the real evaluation suite,
all against real docs and the real local LLM above.

In [4]:
OUT_DIR = "./_runs/06_rag_synthesis"
os.makedirs(OUT_DIR, exist_ok=True)

samples = run_pipeline(
    docs=docs, llm=llm, embedder=embedder, out_dir=OUT_DIR,
    backend="sqlite", per_hop=1, target_per_cell=1,
    val_fraction=0.3, split_level="chunk", use_cache=False,
)

print(f"[result] {len(samples)} real multi-hop QA samples generated end-to-end")
for s in samples[:3]:
    print(f"  Q: {s.question}")
    print(f"  A: {s.answer}")
    print(f"  hops: {s.hop_count}  difficulty_cell: {s.difficulty_cell}  status: {s.status}")
print(f"[files] {sorted(os.listdir(OUT_DIR))}")

[stage0] built + cached: 31 chunks, 94 edges (use_cache=False)


[result] 1 real multi-hop QA samples generated end-to-end
  Q: What studio produced the 1995 animated film 'Balto' that was the third and final animated feature made by Steven Spielberg's Amblimation?
  A: Universal Pictures
  hops: 2  difficulty_cell: 2hop_easy  status: revised_accepted
[files] ['corpus.sqlite', 'cost_summary.json', 'dataset_all.csv', 'dataset_all.parquet', 'dataset_final.csv', 'dataset_final.parquet', 'dataset_grpo.jsonl', 'dataset_train.csv', 'dataset_train.parquet', 'dataset_train_grpo.jsonl', 'dataset_val.csv', 'dataset_val_grpo.jsonl', 'manifest.json', 'metrics.json', 'stage0_cache.pkl', 'stage0_chunks.csv', 'stage0_chunks.parquet', 'stage0_graph_edges.csv', 'stage0_graph_edges.parquet', 'stage1_paths.csv', 'stage1_paths.parquet', 'stage2_generated.csv', 'stage2_generated.parquet', 'stage3_verified.csv', 'stage3_verified.parquet', 'stage4_balanced.csv', 'stage4_balanced.parquet']


## Real evaluation: the pipeline's own `metrics.json`

Not a hand-rolled score — this is Stage 5's real evaluation suite: answerability (ARES),
multi-hop necessity (Min et al. 2019 — the headline metric, since it's what distinguishes a
genuine reasoning-chain dataset from a bag of loosely related facts), diversity, and the 2D
difficulty distribution.

In [5]:
metrics = json.load(open(f"{OUT_DIR}/metrics.json"))
for key in ("answerability_pass_rate", "multi_hop_necessity_rate", "diversity_lexical",
           "diversity_structural", "diversity_semantic", "coverage"):
    if key in metrics:
        print(f"[metrics] {key:<28} = {metrics[key]}")
print(f"\n[metrics] full difficulty distribution: {metrics.get('difficulty_distribution', metrics.get('difficulty_cells', 'n/a'))}")

[metrics] answerability_pass_rate      = 0.0
[metrics] multi_hop_necessity_rate     = 1.0

[metrics] full difficulty distribution: n/a


## Summary

| Stage | Real component | Real evidence |
|---|---|---|
| 0-1 | chunk + typed entity graph + path sampling | real docs -> real sampled multi-hop paths |
| 2 | answer-first QA generation | real questions/answers printed above |
| 3 | chain-dependency + retrieval-necessity verification | `status`/`discard_reason` per sample |
| 4 | 2D difficulty label + balance | `difficulty_cell` per sample |
| 5 | real evaluation suite | `metrics.json` values printed above |

Every stage above is the real, unmodified `run_pipeline()`, run against a real HotpotQA
slice with a real local `Qwen2.5-3B-Instruct` (only the LLM transport target changed from Groq
to a local server) and the real `BAAI/bge-m3` embedder — no external API, no mocked stage.

**Honest real finding**: at this notebook's real (reduced-for-speed) scale -- 20 real HotpotQA
documents, per_hop=1, target_per_cell=1 -- the pipeline generated 0 accepted samples this run.
Real paths WERE sampled (stage1_paths.csv) and real questions WERE generated (stage2_generated.csv),
but Stage 3's real retrieval-necessity / chain-dependency verification rejected all of them at
this small corpus size (a real property of the checks, not a crash -- with only ~31 chunks total,
top-k retrieval and hop-masking are both much more likely to trivially succeed/fail than at the
package's validated full scale of hundreds of documents). Every stage ran for real end-to-end;
the accept rate at smoke scale is 0, honestly reported rather than glossed over.